# 01 - Identifikasi dan Pemeriksaan Dataset

**Proyek:** Analisis Sentimen dan Clustering Ulasan Pelanggan Silol Kopi & Eatery Yogyakarta

Notebook ini menjalankan **Tahap 1** metodologi: memeriksa dataset hasil scraping
untuk mengetahui jumlah data, kelengkapan atribut, keberadaan ulasan kosong,
duplikat, serta sebaran rating.

> Catatan penting: dokumen metodologi menyebut "sekitar 15.000+ ulasan", tetapi
> tidak semua baris berisi teks ulasan. Google Maps tetap mencatat rating tanpa
> komentar, sehingga baris seperti itu tidak dapat dipakai untuk clustering
> berbasis teks. Notebook ini mengukur selisih tersebut secara eksplisit.

In [1]:
import sys
from pathlib import Path

# Menemukan root proyek agar `import src` berjalan dari mana pun notebook dibuka.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)

print("Root proyek:", ROOT)

Root proyek: D:\hikam\tugas DataMining


## 1.1 Memuat dataset mentah

Dataset dibaca dari `data/raw/silol_kopi_eatery_reviews.xlsx`.

In [2]:
from src import io_utils

raw = io_utils.load_raw()
print(f"Ukuran dataset: {raw.shape[0]:,} baris x {raw.shape[1]} kolom")
raw.head(3)

Ukuran dataset: 15,446 baris x 10 kolom


,No,Author,Rating,Date,Timestamp,Review,Owner Response,Language,Review ID,Profile URL
0,1,Rizkya Ajrin,5,setahun lalu,2025-08-17T01:23:16.144Z,"Ramai banget kalo setiap datang kesini, tapi tetap dicariin kursi kok sama pelayannya....","Halo kak Rizkya,<br>Terima kasih atas pengalaman kunjungan nya,di tunggu kunjungan ber...",id,Ci9DQUlRQUNvZENodHljRjlvT21ScVRTMVZjVzB6YzFvd1JFZGpNVGhtTm1KaFdtYxAB,https://www.google.com/maps/contrib/112336004419000863012/reviews?hl=id-ID
1,2,Bita Febianti,1,setahun lalu,2025-08-16T09:26:10.426Z,"Mbah parkiranya nyebelin banget, udah bayar kok ditagih lagi mana ga bantuin narik mot...","Halo Kak Bita,<br>Terima kasih sudah menyampaikan masukan. Mohon maaf atas pengalaman ...",id,Ci9DQUlRQUNvZENodHljRjlvT2toeE5VSmZVVXgxTVV4M1kzZHRXWFY2U1hodE1IYxAB,https://www.google.com/maps/contrib/105447561658976819570/reviews?hl=id-ID
2,3,Nona manis,3,sebulan lalu,2026-09-09T02:11:13.820Z,Pengalaman udh laper..pengen makan bgt...pesen nasgor magelangan...tp rasa nasgornya a...,"Halo kak nona, mohon maaf atas pengalaman yang kurang berkenan. Terima kasih sudah men...",id,Ci9DQUlRQUNvZENodHljRjlvT2poUmVIVldaVzlQVW1ObVV6VjZMV1ZIZURCSFoyYxAB,https://www.google.com/maps/contrib/102953329660431881250/reviews?hl=id-ID


## 1.2 Profil atribut

Tabel berikut merangkum tipe data, jumlah nilai kosong, dan contoh nilai untuk
setiap atribut. Ini menjadi dasar **Tabel 2.1** pada laporan.

In [3]:
from src import audit

overview = audit.dataset_overview(raw)
io_utils.save_csv(overview, config.TABLES_DIR / "01_dataset_overview.csv")
overview

,Atribut,Tipe,Non-Null,Missing,Missing (%),Unique,Contoh Nilai
0,No,int64,15446,0,0.00,15446,1
1,Author,object,15446,0,0.00,15333,Rizkya Ajrin
2,Rating,int64,15446,0,0.00,5,5
3,Date,object,15446,0,0.00,38,setahun lalu
4,Timestamp,object,15446,0,0.00,15446,2025-08-17T01:23:16.144Z
5,Review,object,7267,8179,52.95,6837,"Ramai banget kalo setiap datang kesini, tapi tetap dicariin ..."
6,Owner Response,object,148,15298,99.04,107,"Halo kak Rizkya, Terima kasih atas pengalaman kunjungan nya,..."
7,Language,object,1305,14141,91.55,13,id
8,Review ID,object,15446,0,0.00,15446,Ci9DQUlRQUNvZENodHljRjlvT21ScVRTMVZjVzB6YzFvd1JFZGpNVGhtTm1K...
9,Profile URL,object,15446,0,0.00,15446,https://www.google.com/maps/contrib/112336004419000863012/re...


## 1.3 Kelengkapan data teks ulasan

Pemeriksaan utama: berapa banyak baris yang benar-benar memiliki teks ulasan.

In [4]:
completeness = audit.completeness_summary(raw)
io_utils.save_csv(completeness, config.TABLES_DIR / "01_completeness_summary.csv")
completeness

,Metrik,Nilai
0,Total baris,15446
1,Ulasan dengan teks,7267
2,Ulasan tanpa teks (hanya rating),8179
3,Nilai kosong (NaN) pada kolom Review,8179
4,Duplikat teks ulasan,430
5,Duplikat Review ID,0
6,Ulasan dengan balasan pemilik,148


## 1.4 Sebaran rating

Rating 1-5 bintang dihitung untuk melihat kecenderungan penilaian pelanggan.

In [5]:
rating_dist = audit.rating_distribution(raw)
io_utils.save_csv(rating_dist, config.TABLES_DIR / "01_rating_distribution.csv")
rating_dist

,Rating,Jumlah,Persentase
0,1,524,3.39
1,2,360,2.33
2,3,1201,7.78
3,4,3538,22.91
4,5,9823,63.60
5,Total,15446,100.00


In [6]:
from src import visualize

visualize.plot_rating_distribution(rating_dist)

WindowsPath('D:/hikam/tugas DataMining/results/figures/01_distribusi_rating.png')

## 1.5 Ketersediaan teks per rating

Apakah rating rendah lebih sering disertai komentar? Perbandingan ini penting
karena menentukan seberapa besar data yang hilang bila hanya memakai ulasan berteks.

In [7]:
rating_vs_text = audit.rating_vs_text_crosstab(raw)
io_utils.save_csv(rating_vs_text, config.TABLES_DIR / "01_rating_vs_text.csv")
rating_vs_text

,Rating,Tanpa Teks,Dengan Teks
0,1 bintang,139,385
1,2 bintang,111,249
2,3 bintang,538,663
3,4 bintang,1863,1675
4,5 bintang,5528,4295


In [8]:
visualize.plot_rating_vs_text(rating_vs_text)

WindowsPath('D:/hikam/tugas DataMining/results/figures/01_rating_vs_teks.png')

## 1.6 Panjang ulasan

Ulasan yang sangat pendek (mis. "ok", "👍") tidak informatif untuk clustering.

In [9]:
length_stats = audit.review_length_stats(raw)
io_utils.save_csv(length_stats, config.TABLES_DIR / "01_review_length_stats.csv")
length_stats

,Statistik,Panjang Karakter
0,count,7267.00
1,mean,102.25
2,std,156.06
3,min,1.00
4,25%,24.00
5,50%,52.00
6,75%,116.00
7,max,4104.00


In [10]:
review_text = raw["Review"].astype("string").fillna("").str.strip()
visualize.plot_review_length(review_text[review_text.ne("")].str.len())

WindowsPath('D:/hikam/tugas DataMining/results/figures/01_panjang_ulasan.png')

## 1.7 Tren ulasan dari waktu ke waktu

Kolom `Timestamp` menyimpan waktu unggah sehingga bisa dipakai untuk analisis tren.

In [11]:
trend = audit.yearly_trend(raw)
io_utils.save_csv(trend, config.TABLES_DIR / "01_yearly_trend.csv")
trend

,Tahun,Jumlah,Rating_Rata2
0,2017,55,1.927
1,2018,4418,4.241
2,2019,4365,4.494
3,2020,2048,4.511
4,2021,1153,4.589
5,2022,1173,4.555
6,2023,879,4.457
7,2024,596,4.341
8,2025,446,4.298
9,2026,313,4.355


In [12]:
visualize.plot_yearly_trend(trend)

WindowsPath('D:/hikam/tugas DataMining/results/figures/01_tren_tahunan.png')

## 1.8 Sebaran bahasa

Kolom `Language` pada hasil scraping menandai bahasa ulasan. Sebagian besar
ulasan berbahasa Indonesia, sisanya Inggris dan bahasa lain.

In [13]:
lang_dist = audit.language_distribution(raw)
io_utils.save_csv(lang_dist, config.TABLES_DIR / "01_language_distribution.csv")
lang_dist.head(10)

,Bahasa,Jumlah,Persentase
0,(kosong),14141,91.55
1,id,1090,7.06
2,en,189,1.22
3,ms,9,0.06
4,jv,5,0.03
5,nl,2,0.01
6,ar,2,0.01
7,tr,2,0.01
8,zh,1,0.01
9,es,1,0.01


## 1.9 Pembersihan awal dataset

Tiga aturan diterapkan berurutan:

1. Buang baris tanpa teks ulasan (hanya rating).
2. Buang duplikat teks (setelah dinormalisasi spasi dan huruf kecil).
3. Buang ulasan yang terlalu pendek (< 3 karakter).

In [14]:
clean, recap = audit.build_clean_dataset(raw)
io_utils.save_csv(recap, config.TABLES_DIR / "01_cleaning_recap.csv")
io_utils.save_csv(clean, config.CLEAN_CSV)
recap

,Tahap,Jumlah Ulasan,Dibuang
0,Dataset mentah,15446,0
1,Setelah membuang ulasan tanpa teks,7267,8179
2,Setelah membuang duplikat teks,6798,469
3,Setelah membuang ulasan terlalu pendek,6795,3


In [15]:
visualize.plot_completeness(recap)

WindowsPath('D:/hikam/tugas DataMining/results/figures/01_tahapan_pembersihan.png')

## 1.10 Kesimpulan Tahap 1

Ringkasan otomatis angka kunci:

In [16]:
ringkasan = pd.DataFrame(
    {
        "Indikator": [
            "Total baris dataset",
            "Baris dengan teks ulasan",
            "Baris tanpa teks (hanya rating)",
            "Duplikat teks",
            "Ulasan siap analisis",
            "Rating rata-rata (data bersih)",
            "Rentang tanggal",
            "Jumlah pengulas unik",
        ],
        "Nilai": [
            f"{len(raw):,}",
            f"{int(raw['Review'].notna().sum()):,}",
            f"{int(raw['Review'].isna().sum()):,}",
            f"{int(raw['Review'].duplicated().sum()):,}",
            f"{len(clean):,}",
            f"{clean['Rating'].mean():.3f}",
            f"{pd.to_datetime(raw['Timestamp'], errors='coerce', utc=True).min():%Y-%m-%d} s.d. "
            f"{pd.to_datetime(raw['Timestamp'], errors='coerce', utc=True).max():%Y-%m-%d}",
            f"{clean['Author'].nunique():,}",
        ],
    }
)
io_utils.save_csv(ringkasan, config.TABLES_DIR / "01_ringkasan_tahap1.csv")
ringkasan

,Indikator,Nilai
0,Total baris dataset,"15,446"
1,Baris dengan teks ulasan,"7,267"
2,Baris tanpa teks (hanya rating),"8,179"
3,Duplikat teks,"8,608"
4,Ulasan siap analisis,"6,795"
5,Rating rata-rata (data bersih),4.252
6,Rentang tanggal,2017-12-16 s.d. 2026-10-14
7,Jumlah pengulas unik,"6,766"


### Catatan untuk laporan

- Dataset mentah berisi ribuan baris, namun **hanya sebagian yang memiliki teks**
  sehingga jumlah ulasan yang benar-benar dianalisis lebih kecil.
- Selisih ini bukan kesalahan scraping, melainkan karakteristik data Google Maps
  yang mengizinkan rating tanpa komentar.
- Data bersih disimpan di `data/processed/01_reviews_clean.csv` dan dipakai
  notebook berikutnya.